# RFP Evaluation & Supplier Ranking

Runs the same code as the Streamlit app without the UI: set up the database, score the four sample proposals, check the formulas by hand and run the tests. Leave the key blank to use the offline mock scorer.

In [ ]:
import os
import pathlib
import sys
import zipfile

PROJECT = "rfp-evaluation"
if not pathlib.Path("app.py").exists():
    if not pathlib.Path(PROJECT).exists():
        try:
            from google.colab import files

            uploaded = files.upload()
            zipfile.ZipFile(next(iter(uploaded))).extractall(".")
        except ImportError:
            raise SystemExit(f"Unzip {PROJECT}.zip next to this notebook first.") from None
    os.chdir(PROJECT)
sys.path.insert(0, os.getcwd())
print("Working in", os.getcwd())

In [ ]:
!{sys.executable} -m pip install -q -r requirements.txt

In [ ]:
import getpass

from rfp_evaluation.evaluate import get_settings

key = getpass.getpass("LITELLM_API_KEY (blank = offline mock mode): ").strip()
if key:
    os.environ["LITELLM_API_KEY"] = key
    os.environ["LITELLM_API_BASE"] = input("LITELLM_API_BASE (blank = call the provider directly): ").strip()
    os.environ["LITELLM_MODEL"] = input("Model [groq/openai/gpt-oss-120b]: ").strip() or "groq/openai/gpt-oss-120b"
    os.environ["LLM_MODE"] = "live"
else:
    os.environ["LLM_MODE"] = "mock"
os.environ["RFP_DB_PATH"] = "data/notebook.db"
print("model:", get_settings().label)

## Criteria

In [ ]:
import pandas as pd
from rfp_evaluation import store

store.setup(reset_criteria=True)
pd.DataFrame(store.get_criteria())[["criterion_id", "name", "weight", "max_score", "is_active"]]

## Score the four sample proposals

In [ ]:
from rfp_evaluation.run import run_evaluation
from rfp_evaluation.validate import Proposal
from rfp_evaluation.export import leaderboard_table

samples = [
    ("Meridian Softworks", "2026-09-10", 4.5, "Meridian_Softworks_Technical_and_Commercial_Proposal.pdf"),
    ("QuickDesk Solutions", "2026-09-08", 2.0, "QuickDesk - Proposal for Sundaram Retail.pdf"),
    ("Tarang Systems", "2026-09-09", 4.0, "Tarang Systems_Response_SRL-IT-2026-014.pdf"),
    ("Northstar Consulting", "2026-09-11", 5.0, "NSCS_Proposal_Final (signed).pdf"),
]
proposals = [Proposal(n, d, e, f, pathlib.Path("data/proposals", f).read_bytes()) for n, d, e, f in samples]
run = run_evaluation(proposals, "Customer support platform - tender SRL/IT/2026/014", progress=print)

leaderboard_table(run)

In [ ]:
pd.DataFrame(run["suppliers"][-1]["criteria"])[
    ["name", "weight", "score", "benchmark", "gap", "relative_pct", "evidence_page", "evidence"]
]

## Check the formulas by hand

In [ ]:
for s in run["suppliers"]:
    absolute = sum(c["score"] / c["max_score"] * c["weight"] for c in s["criteria"])
    ppi = sum(
        c["score"] / c["benchmark"] * 100 * c["weight"] if c["benchmark"] else 0 for c in s["criteria"]
    ) / sum(c["weight"] for c in s["criteria"])
    print(
        f"{s['supplier_name']:<22} absolute {absolute:6.2f} (app {s['absolute_score']:6.2f})"
        f"  PPI {ppi:6.2f} (app {s['ppi']:6.2f})"
    )

## Bad inputs are rejected before any LLM call

In [ ]:
try:
    run_evaluation(
        [
            Proposal("Meridian", "2030-01-01", 9, "a.pdf", proposals[0].data),
            Proposal("meridian", "2026-09-01", 3, "notes.txt", b"hi"),
        ]
    )
except ValueError as exc:
    print(exc)

## What-if and reports

In [ ]:
from rfp_evaluation.rank import rank_with_weights

[
    (s["final_rank"], s["supplier_name"], s["ppi"])
    for s in rank_with_weights(run["suppliers"], {1: 10, 2: 40, 3: 10, 4: 10, 5: 30})
]

In [ ]:
from rfp_evaluation.export import pdf_report, excel_workbook

pathlib.Path("data/notebook_report.pdf").write_bytes(pdf_report(run))
pathlib.Path("data/notebook_scores.xlsx").write_bytes(excel_workbook(run))
print("reports written to data/")

## Tests

## Blank and corrupted PDFs score 0 with a warning

In [ ]:
bad = [
    Proposal(
        "Blank Co",
        "2026-09-01",
        3,
        "blank.pdf",
        pathlib.Path("data/bad_files/Scan_20260911_0932.pdf").read_bytes(),
    ),
    Proposal(
        "Broken Ltd",
        "2026-09-01",
        3,
        "broken.pdf",
        pathlib.Path("data/bad_files/Proposal_upload_final.pdf").read_bytes(),
    ),
]
result = run_evaluation(proposals[:2] + bad, "Bad files")
print(*result["warnings"], sep="\n")
leaderboard_table(result)

In [ ]:
!{sys.executable} -m pytest -q

Run the app with `streamlit run app.py`.